# ⚡ EducationSwap Headless Runner (No Web Interface Needed!)
**Zero disconnection issues.** Everything runs directly in this Google Colab notebook:
1. **Upload your Source (Face) and Target (Image/Video) directly to Colab or Google Drive**.
2. **Configure options (Face Swapper, Enhancer, Masker, etc.) with clean interactive Colab Form inputs**.
3. **Run headless at maximum CUDA GPU speed with 100% stable live progress logs**.
4. **View result & auto-save to Google Drive or download directly**.

## 1. Keep-Alive Heartbeat (Optional)
Keeps the Colab session active in background tabs.

In [ ]:
from IPython.display import display, HTML

display(HTML('''
<div style="padding:12px; border:2px solid #22c55e; border-radius:8px; background:#052e16; color:#86efac; font-family:sans-serif;">
  <b>🟢 Anti-Disconnect Heartbeat Active:</b> Keeps Colab runtime alive.
</div>
<script>
function keepAlive() {
  try {
    let ctx = new (window.AudioContext || window.webkitAudioContext)();
    let osc = ctx.createOscillator();
    let gain = ctx.createGain();
    gain.gain.value = 0.0001;
    osc.connect(gain);
    gain.connect(ctx.destination);
    osc.start();
  } catch(e) {}
  setInterval(() => {
    let btn = document.querySelector('colab-connect-button');
    if (btn) btn.click();
  }, 60000);
}
keepAlive();
</script>
'''))


## 2. Mount Google Drive
Mounts your Google Drive. All swapped results will be auto-saved to `/content/drive/MyDrive/EducationSwap/outputs`.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/EducationSwap/inputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/EducationSwap/outputs', exist_ok=True)
print("✅ Google Drive connected!")
print("📁 Inputs folder:  /content/drive/MyDrive/EducationSwap/inputs")
print("📁 Outputs folder: /content/drive/MyDrive/EducationSwap/outputs")


## 3. Clone Repository
Pulls the latest EducationSwap code.

In [ ]:
!git clone https://github.com/hasnatvip/swpeducation.git
%cd swpeducation


## 4. Install Dependencies & NVIDIA Libraries

In [ ]:
!pip install nvidia-cublas-cu12 nvidia-cudnn-cu12==8.9.2.26 nvidia-cufft-cu12 nvidia-curand-cu12 nvidia-cusolver-cu12 nvidia-cusparse-cu12 nvidia-nvtx-cu12
!python install.py --skip-conda cuda@12


## 5. Upload Files Directly via Colab (Or Place in Google Drive)
Run this cell to upload your **Source (Face)** image and **Target (Video/Image)** file straight from your computer into Colab.

In [ ]:
from google.colab import files
import os

os.makedirs('/content/inputs', exist_ok=True)

print("📤 [1/2] Select and upload your SOURCE face image (JPG/PNG):")
uploaded_source = files.upload()
source_file = list(uploaded_source.keys())[0] if uploaded_source else ""

print("\n📤 [2/2] Select and upload your TARGET image or video (MP4/JPG/PNG):")
uploaded_target = files.upload()
target_file = list(uploaded_target.keys())[0] if uploaded_target else ""

if source_file and target_file:
    print(f"\n✅ Files uploaded successfully:")
    print(f"   Source: /content/swpeducation/{source_file}")
    print(f"   Target: /content/swpeducation/{target_file}")
else:
    print("ℹ️ You can also manually put files in Google Drive at /content/drive/MyDrive/EducationSwap/inputs/")


## 6. Run Headless Face Swap with Max GPU Settings (Colab Form UI)
Configured for **maximum CUDA GPU throughput & 100% GPU utilization**:
- **16 Threads (up to 32)**: Keeps GPU compute saturated without idle gaps.
- **Tolerant Memory**: Caches all models in VRAM for zero re-allocation latency.
- **In-Memory Workflow**: Bypasses slow Colab virtual disk writes.
- **NVENC Hardware Acceleration**: GPU video encoding instead of slow CPU encoding.
- **Single-angle Face Detection**: Cuts redundant rotational checks for 4x detection speed.

In [ ]:
#@title ⚙️ Headless Swap Settings (Max GPU Optimized)
import os
import site
import glob
import subprocess

# File paths (Edit these or use the uploaded files)
SOURCE_PATH = "" #@param {type:"string"}
TARGET_PATH = "" #@param {type:"string"}
OUTPUT_NAME = "swapped_result.mp4" #@param {type:"string"}

# Processors
ENABLE_FACE_SWAPPER = True #@param {type:"boolean"}
ENABLE_FACE_ENHANCER = True #@param {type:"boolean"}

# Model Choices
SWAPPER_MODEL = "hyperswap_1a_256" #@param ["hyperswap_1a_256", "inswapper_128", "simswap_256", "simswap_512_unofficial", "ghost_1_256", "ghost_2_256", "ghost_3_256", "uniface_256", "blendswap_256"]
ENHANCER_MODEL = "gfpgan_1.4" #@param ["gfpgan_1.4", "codeformer", "restoreformer_plus_plus", "gpen_bfr_256", "gpen_bfr_512"]
ENHANCER_BLEND = 80 #@param {type:"slider", min:0, max:100, step:5}

# ⚡ Performance & Maximum GPU Utilization Settings
EXECUTION_THREADS = 16 #@param {type:"slider", min:1, max:32, step:1}
VIDEO_MEMORY_STRATEGY = "tolerant" #@param ["tolerant", "moderate", "strict"]
WORKFLOW_STRATEGY = "memory" #@param ["memory", "disk"]
FACE_DETECTOR_MODEL = "yolo_face" #@param ["yolo_face", "scrfd", "yunet", "retinaface"]
OUTPUT_VIDEO_ENCODER = "auto_detect_gpu" #@param ["auto_detect_gpu", "h264_nvenc", "hevc_nvenc", "libx264"]
OUTPUT_VIDEO_PRESET = "fast" #@param ["ultrafast", "superfast", "veryfast", "faster", "fast", "medium"]
VIDEO_QUALITY = 80 #@param {type:"slider", min:1, max:100, step:1}

# Auto-detect paths if blank
if not SOURCE_PATH:
    if 'source_file' in locals() and source_file:
        SOURCE_PATH = f"/content/swpeducation/{source_file}"
    else:
        # Check Google Drive inputs
        g_inputs = glob.glob("/content/drive/MyDrive/EducationSwap/inputs/*")
        if g_inputs:
            SOURCE_PATH = g_inputs[0]

if not TARGET_PATH:
    if 'target_file' in locals() and target_file:
        TARGET_PATH = f"/content/swpeducation/{target_file}"
    else:
        g_inputs = glob.glob("/content/drive/MyDrive/EducationSwap/inputs/*")
        if len(g_inputs) > 1:
            TARGET_PATH = g_inputs[1]

assert SOURCE_PATH and os.path.exists(SOURCE_PATH), f"❌ SOURCE_PATH not found: {SOURCE_PATH}"
assert TARGET_PATH and os.path.exists(TARGET_PATH), f"❌ TARGET_PATH not found: {TARGET_PATH}"

# Build output path (prioritize Google Drive)
output_dir = "/content/drive/MyDrive/EducationSwap/outputs" if os.path.exists("/content/drive/MyDrive") else "/content/swpeducation"
output_full_path = os.path.join(output_dir, OUTPUT_NAME)

# Build processors list
procs = []
if ENABLE_FACE_SWAPPER:
    procs.append("face_swapper")
if ENABLE_FACE_ENHANCER:
    procs.append("face_enhancer")

# Setup CUDA LD_LIBRARY_PATH and GPU Performance Environment Variables
site_packages = site.getsitepackages()[0]
nvidia_libs = glob.glob(f"{site_packages}/nvidia/*/lib")
os.environ['LD_LIBRARY_PATH'] = ":".join(nvidia_libs) + ":" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'
os.environ['ORT_CUDA_GEMM_OPTIONS'] = '1'
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '1'

# Resolve video encoder (prioritize NVIDIA NVENC for fast GPU encoding)
resolved_encoder = "libx264"
if OUTPUT_VIDEO_ENCODER == "auto_detect_gpu":
    try:
        encoders_out = subprocess.check_output(["ffmpeg", "-encoders"], stderr=subprocess.DEVNULL, text=True)
        if "h264_nvenc" in encoders_out:
            resolved_encoder = "h264_nvenc"
    except Exception:
        pass
else:
    resolved_encoder = OUTPUT_VIDEO_ENCODER

# Assemble optimized execution command
cmd = [
    "python", "-u", "educationswap.py", "headless-run",
    "-s", SOURCE_PATH,
    "-t", TARGET_PATH,
    "-o", output_full_path,
    "--execution-providers", "cuda",
    "--execution-thread-count", str(EXECUTION_THREADS),
    "--video-memory-strategy", str(VIDEO_MEMORY_STRATEGY),
    "--workflow-strategy", str(WORKFLOW_STRATEGY),
    "--face-detector-model", str(FACE_DETECTOR_MODEL),
    "--face-detector-angles", "0",
    "--output-video-encoder", resolved_encoder,
    "--output-video-preset", str(OUTPUT_VIDEO_PRESET),
    "--output-video-quality", str(VIDEO_QUALITY),
    "--log-level", "info"
]

if procs:
    cmd.extend(["--processors"] + procs)
if ENABLE_FACE_SWAPPER:
    cmd.extend(["--face-swapper-model", SWAPPER_MODEL])
if ENABLE_FACE_ENHANCER:
    cmd.extend(["--face-enhancer-model", ENHANCER_MODEL, "--face-enhancer-blend", str(ENHANCER_BLEND)])

print("=" * 65)
print(f"🚀 Starting Headless Face Swap (Max GPU Mode):")
print(f"   Source:         {SOURCE_PATH}")
print(f"   Target:         {TARGET_PATH}")
print(f"   Output:         {output_full_path}")
print(f"   Threads:        {EXECUTION_THREADS} (Parallel CUDA workers)")
print(f"   Workflow:       {WORKFLOW_STRATEGY} (Zero disk latency)")
print(f"   Memory Mode:    {VIDEO_MEMORY_STRATEGY} (Full VRAM caching)")
print(f"   Video Encoder:  {resolved_encoder}")
print(f"   Processors:     {', '.join(procs)}")
print("=" * 65)

# Run with live unbuffered output
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0 and os.path.exists(output_full_path):
    print(f"\n🎉 SUCCESS! Result saved at: {output_full_path}")
    from IPython.display import Video, Image, display
    if output_full_path.lower().endswith(('.mp4', '.mov')):
        display(Video(output_full_path, embed=True, width=600))
    elif output_full_path.lower().endswith(('.png', '.jpg', '.jpeg')):
        display(Image(output_full_path, width=600))
else:
    print("\n❌ Swap process encountered an error.")


## 7. Download Finished File to Your Computer (Optional)
If you want to download the result file directly to your Mac without opening Google Drive.

In [ ]:
from google.colab import files
import os

if 'output_full_path' in locals() and os.path.exists(output_full_path):
    print(f"Downloading {output_full_path}...")
    files.download(output_full_path)
else:
    print("No output file found. Run Cell 6 first.")
